In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#%matplotlib inline

#import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
from celloracle import motif_analysis as ma
import celloracle as co


/home/junyichen/anaconda3/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
df_cire = pd.read_csv('/data2st1/junyi/output/atac1112/cicre/HIP_HPF_Subiculum_IT_Glut_ALL_circe_network.csv',index_col=0)

In [3]:
df_peaks = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/cCRE_annotated.csv')

In [ ]:
df_peaks

In [4]:
df_cire_selected = df_cire[df_cire.pval_adj<0.05]

In [5]:
df_cire_selected = df_cire_selected.rename({"score":"coaccess"},axis=1)

In [ ]:
df_cire_selected

In [6]:
peaks = df_peaks.names.str.replace("[:-]","_")

In [ ]:
peaks

In [7]:
tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome="mm10")

# Check results
tss_annotated.tail()


que bed peaks: 1667332
tss peaks in que: 45233


,chr,start,end,gene_short_name,strand
45228,chr4,129491491,129491992,Fam229a,-
45229,chr4,129490946,129491447,Fam229a,-
45230,chr4,129490400,129490901,Fam229a,-
45231,chr17,24472423,24472924,Bricd5,+
45232,chr5,21035659,21036160,Ptpn12,-


In [9]:
integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated,
                                               cicero_connections=df_cire_selected)
print(integrated.shape)
integrated.head()


(80480, 3)


,peak_id,gene_short_name,coaccess
0,chr10_100015529_100016030,Kitl,1.00000
1,chr10_100016238_100016739,Kitl,0.19095
2,chr10_100487209_100487710,Tmtc3,1.00000
3,chr10_100488034_100488535,Cep290,1.00000
4,chr10_100488034_100488535,Tmtc3,1.00000


In [10]:
integrated

,peak_id,gene_short_name,coaccess
0,chr10_100015529_100016030,Kitl,1.00000
1,chr10_100016238_100016739,Kitl,0.19095
2,chr10_100487209_100487710,Tmtc3,1.00000
3,chr10_100488034_100488535,Cep290,1.00000
4,chr10_100488034_100488535,Tmtc3,1.00000
...,...,...,...
80475,chrY_1246041_1246542,Uty,1.00000
80476,chrY_1286369_1286870,Ddx3y,1.00000
80477,chrY_896718_897219,Kdm5d,1.00000
80478,chrY_897281_897782,Kdm5d,1.00000


In [11]:
# PLEASE make sure reference genome is correct.
ref_genome = "mm10"

genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
print(ref_genome, "installation: ", genome_installation)


genome mm10 is not installed in this environment.
Please install genome using genomepy.
e.g.
    >>> import genomepy
    >>> genomepy.install_genome(name="mm10", provider="UCSC")
mm10 installation:  False


In [12]:
if not genome_installation:
    import genomepy
    genomepy.install_genome(name=ref_genome, provider="UCSC", genomes_dir=None)
else:
    print(ref_genome, "is installed.")

16:26:09 | INFO | Downloading assembly summaries from UCSC
16:26:15 | INFO | Downloading genome from UCSC. Target URL: https://hgdownload.soe.ucsc.edu/goldenPath/mm10/bigZips/chromFa.tar.gz...


Download:   0%|          | 0.00/830M [00:00<?, ?B/s]

16:27:33 | INFO | Genome download successful, starting post processing...
16:28:04 | INFO | name: mm10
16:28:04 | INFO | local name: mm10
16:28:04 | INFO | fasta: /home/junyichen/.local/share/genomes/mm10/mm10.fa


Filtering Fasta: 0.00 lines [00:00, ? lines/s]

In [18]:
integrated=integrated[integrated.coaccess>0.5]

In [25]:
tfi = ma.TFinfo(peak_data_frame=integrated,
                ref_genome=ref_genome,
                genomes_dir=None)


In [26]:
%%time
# Scan motifs. !!CAUTION!! This step may take several hours if you have many peaks!
tfi.scan(fpr=0.02,
         motifs=None,  # If you enter None, default motifs will be loaded.
         verbose=True)

# Save tfinfo object
tfi.to_hdf5(file_path="/data2st1/junyi/output/test/test1.celloracle.tfinfo")


No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2026-01-02 16:42:20,929 - DEBUG - using background: genome mm10 with size 200


TypeError: Scanner.set_background() got an unexpected keyword argument 'length'